In [1]:
# Experiment: testing different max_depth values for the Random Forest
# This notebook lives in experiments/setup/, which is TWO levels deep,
# so data paths use ../../ to reach the project root.

import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score, average_precision_score, precision_score, recall_score, confusion_matrix

# Load the processed data (note the ../../ - two levels up from experiments/setup/)
X_train = pd.read_csv("../../data/processed_fe/X_train_balanced.csv")
y_train = pd.read_csv("../../data/processed_fe/y_train_balanced.csv").values.ravel()
X_test = pd.read_csv("../../data/processed_fe/X_test.csv")
y_test = pd.read_csv("../../data/processed_fe/y_test.csv").values.ravel()

print("Data loaded successfully.")
print("Training set:", X_train.shape, "| Test set:", X_test.shape)

Data loaded successfully.
Training set: (454902, 31) | Test set: (56962, 31)


In [2]:
# Test several max_depth values and record performance for each
# We use None (unlimited), 10, 20, and 30 to see how depth affects the results
depth_values = [None, 10, 20, 30]
results = []

for depth in depth_values:
    # Train a Random Forest with this depth
    model = RandomForestClassifier(
        n_estimators=100, max_depth=depth, n_jobs=-1, random_state=42
    )
    model.fit(X_train, y_train)

    # Predict on the test set
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]

    # Record the key metrics
    cm = confusion_matrix(y_test, y_pred)
    results.append({
        "max_depth": "None" if depth is None else depth,
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1": round(f1_score(y_test, y_pred), 4),
        "PR_AUC": round(average_precision_score(y_test, y_proba), 4),
        "False_Positives": cm[0, 1],
        "Fraud_Caught": cm[1, 1],
    })
    print(f"Finished max_depth = {depth}")

# Build a results table
results_df = pd.DataFrame(results)
print("\nExperiment complete.")
results_df

Finished max_depth = None
Finished max_depth = 10
Finished max_depth = 20
Finished max_depth = 30

Experiment complete.


,max_depth,Precision,Recall,F1,PR_AUC,False_Positives,Fraud_Caught
0,None,0.8901,0.8265,0.8571,0.8799,10,81
1,10,0.4200,0.8571,0.5638,0.8171,116,84
2,20,0.7523,0.8367,0.7923,0.8639,27,82
3,30,0.8421,0.8163,0.8290,0.8766,15,80


In [3]:
import os

# Save the experiment results to the results folder
os.makedirs("../results", exist_ok=True)
results_df.to_csv("../results/max_depth_experiment_results.csv", index=False)
print("Results saved to experiments/results/max_depth_experiment_results.csv")

Results saved to experiments/results/max_depth_experiment_results.csv
